# 视频 → 人体姿态 JSON / NPZ → Blender

从上到下执行：配置 → 克隆本项目 → pip/uv 安装依赖 → 克隆并补丁 SAM/DINOv3 → 下载模型 → 检查与提取。
准备步骤直接执行命令，不需要另外调用安装/克隆/下载封装函数。命令通过 `subprocess.run(..., check=True)` 执行，失败会停止当前单元，路径含空格也可使用。

使用 Python 3.11（也支持 3.12）kernel，先按 README 准备匹配硬件的 PyTorch/torchvision。默认 GPU、多人、前 30 个采样帧；CPU 请将 `DEVICE` 改为 `cpu`。
每个准备单元执行时才会联网或安装；复用已有资源时会校验，不执行 pull/reset。安装后重启 kernel，再运行配置和项目定位单元，跳过依赖安装单元继续。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# 在项目根目录/extractor 目录运行时直接复用；只上传此 notebook 时克隆到当前目录。
PROJECT_URL = "https://github.com/fjlikun/pose_pipeline.git"
PROJECT_BRANCH = "develop"  # 默认仅克隆不含 MP4 的代码分支
WORK_DIR = Path.cwd().resolve()
if (WORK_DIR / "extractor" / "extract_video.py").is_file():
    PROJECT_DIR = WORK_DIR
elif (WORK_DIR / "extract_video.py").is_file() and WORK_DIR.name == "extractor":
    PROJECT_DIR = WORK_DIR.parent
else:
    PROJECT_DIR = WORK_DIR / "pose_pipeline"
EXTRACTOR_DIR = PROJECT_DIR / "extractor"
RUNTIME_DIR = PROJECT_DIR / "runtime"
PACKAGE_MANAGER = "pip"  # 可改为 "uv"；uv 命令需要预先安装
DEVICE = "cuda"          # CPU: "cpu"，后面的源码单元会应用 CPU 补丁
MULTI_PERSON = True

VIDEO = PROJECT_DIR / "input.mp4"  # 改成自己的视频绝对路径
OUTPUT_DIR = EXTRACTOR_DIR / "motion_multi"  # 多人输出必须是新目录
OUTPUT_JSON = EXTRACTOR_DIR / "motion_single.json"
SAM_REPO = RUNTIME_DIR / "sam-3d-body"
DINOV3_REPO = RUNTIME_DIR / "dinov3"
CHECKPOINT_DIR = RUNTIME_DIR / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "model.ckpt"
MHR_MODEL = CHECKPOINT_DIR / "assets" / "mhr_model.pt"
DETECTOR_WEIGHTS = RUNTIME_DIR / "detector" / "fasterrcnn_resnet50_fpn_v2_coco.pth"
MODEL_SOURCE = "modelscope"  # 或 "huggingface"；官方源需提前获得访问权限
MODEL_REVISION = "master" if MODEL_SOURCE == "modelscope" else "main"

STRIDE = 1
MAX_FRAMES = 30  # 正式全视频改为 None
START_SECONDS = 0.0
END_SECONDS = None
DETECTOR_THRESHOLD = 0.7
MAX_PEOPLE = 20
TRACK_IOU_THRESHOLD = 0.25
MAX_MISSING_SECONDS = 1.0  # 应大于正常采样间隔 STRIDE / 源 FPS
SAVE_NPZ = True
NPZ_COMPRESSED = True
INFERENCE_TYPE = "body"
CONTINUE_ON_POSE_ERROR = False
SINGLE_BBOX = None  # 单人模式: [x1, y1, x2, y2]；None 为全图
print("Python:", sys.executable)
print("项目:", PROJECT_DIR)

## 1. git clone 本项目

已有完整项目时直接复用；空环境中仅浅克隆 `PROJECT_BRANCH`（默认 develop）：`git clone --branch develop --single-branch --depth 1`。如果仓库为私有，需要预先配置 Git 访问权限，或修改 `PROJECT_URL`。develop 不包含 MP4 或 Git LFS 资源规则，推理使用上面指定的自己的视频。

In [ ]:
if not (EXTRACTOR_DIR / "extract_video.py").is_file():
    if PROJECT_DIR.exists():
        raise FileExistsError(f"已有目录不是完整项目，请修改 PROJECT_DIR: {PROJECT_DIR}")
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", PROJECT_BRANCH, "--single-branch",
                    "--depth", "1", PROJECT_URL, str(PROJECT_DIR)], check=True)
if not (EXTRACTOR_DIR / "requirements.txt").is_file():
    raise FileNotFoundError(EXTRACTOR_DIR / "requirements.txt")
if str(EXTRACTOR_DIR) not in sys.path:
    sys.path.insert(0, str(EXTRACTOR_DIR))
os.environ["SAM3D_DINOV3_REPO"] = str(DINOV3_REPO)
print("使用项目:", PROJECT_DIR)

## 2. 通过 requirements.txt 安装依赖（pip / uv）

`extractor/requirements.txt` 包含固定 SAM 版本的 Python 依赖、PyAV、NumPy、Notebook 及模型下载客户端。它不安装 PyTorch/torchvision/CUDA，请先选好已有运行环境。

实际命令为 `当前Python -m pip install -r requirements.txt -c torch-constraints.txt`，或 `uv pip install --python 当前Python -r requirements.txt -c torch-constraints.txt`。
下面固定现有 torch/CUDA 相关包版本，其他依赖仍按 pip/uv 正常解析，可能调整版本；依赖冲突会直接报错。

安装后重启 kernel，再执行配置和项目定位单元，然后跳过本安装单元继续。某些平台的 xtcocotools 等可能需要 C/C++ 编译工具，详见 README。

In [ ]:
from importlib.metadata import distributions
import re
import shutil

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError("请选择 Python 3.11/3.12 kernel")
versions = {}
for dist in distributions():
    if dist.metadata.get("Name"):
        name = re.sub(r"[-_.]+", "-", dist.metadata["Name"]).lower()
        versions[name] = dist.version
if not all(name in versions for name in ("torch", "torchvision")):
    raise RuntimeError("请先按 README 安装匹配 CPU/CUDA 的 torch 和 torchvision，再执行本单元")
if PACKAGE_MANAGER == "pip":
    install_cmd = [sys.executable, "-m", "pip", "install"]
    check_cmd = [sys.executable, "-m", "pip", "check"]
elif PACKAGE_MANAGER == "uv":
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("未找到 uv；请先安装 uv，或设置 PACKAGE_MANAGER='pip'")
    install_cmd = [uv, "pip", "install", "--python", sys.executable]
    check_cmd = [uv, "pip", "check", "--python", sys.executable]
else:
    raise ValueError("PACKAGE_MANAGER 必须为 pip 或 uv")
protected = {name: version for name, version in versions.items()
             if name in {"torch", "torchvision", "torchaudio", "torchtext", "torchdata", "triton", "cuda", "cudnn", "nvidia", "pytorch"}
             or name.startswith(("nvidia-", "cuda-", "cudnn", "pytorch-triton", "torch-"))}
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
constraints = RUNTIME_DIR / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in sorted(protected.items())), encoding="utf-8")
command = [*install_cmd, "-r", str(EXTRACTOR_DIR / "requirements.txt"), "-c", str(constraints)]
print(subprocess.list2cmdline(command))
subprocess.run(command, check=True)
subprocess.run(check_cmd, check=True)
print("依赖安装完成。请重启 kernel，重跑配置和项目定位单元，再跳过本单元继续。")

## 3. git clone 固定版本的 SAM / DINOv3

首次执行 `git clone --no-checkout` 和 `git checkout --detach <commit>`。已有目录只核对官方 origin、commit 和修改，不 pull、不 reset。

In [ ]:
from runtime_checks import SAM_URL, SAM_COMMIT, DINO_URL, DINO_COMMIT

for repo, url, commit in ((SAM_REPO, SAM_URL, SAM_COMMIT), (DINOV3_REPO, DINO_URL, DINO_COMMIT)):
    if repo.exists():
        if not (repo / ".git").exists():
            raise RuntimeError(f"不是 git checkout，请选择新路径: {repo}")
    else:
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", url, str(repo)], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    if actual != commit or origin.rstrip("/") != url:
        raise RuntimeError(f"已有源码版本或 origin 不匹配，请选择新目录: {repo}")
if subprocess.check_output(["git", "-C", str(DINOV3_REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
    raise RuntimeError("DINOv3 有 tracked 修改，请检查或使用新目录")
print("SAM 和 DINOv3 固定版本已就绪")

### 应用本地 DINOv3 / checkpoint 补丁；CPU 自动追加设备补丁

先执行 `git apply --check`，成功再 `git apply`。已应用相同补丁时跳过；未知修改报错。`DEVICE='cpu'` 时再应用 `cpu_device.patch`，仅修改模型设备选择，不安装 CPU torch。改变源码设备补丁后请重启 kernel。

In [ ]:
from runtime_checks import normalized_patch

if DEVICE not in ("cpu", "cuda"):
    raise ValueError("DEVICE 必须为 cpu 或 cuda")
gpu_patch = EXTRACTOR_DIR / "gpu_local_dinov3.patch"
full_patch = EXTRACTOR_DIR / "cpu_local_dinov3.patch"
current = normalized_patch(subprocess.check_output(["git", "-C", str(SAM_REPO), "diff", "HEAD"], text=True))
if current not in ("", normalized_patch(gpu_patch.read_text(encoding="utf-8")), normalized_patch(full_patch.read_text(encoding="utf-8"))):
    raise RuntimeError("SAM 有未知修改，停止应用补丁")
if not current:
    subprocess.run(["git", "-C", str(SAM_REPO), "apply", "--check", str(gpu_patch)], check=True)
    subprocess.run(["git", "-C", str(SAM_REPO), "apply", str(gpu_patch)], check=True)
if DEVICE == "cpu":
    current = normalized_patch(subprocess.check_output(["git", "-C", str(SAM_REPO), "diff", "HEAD"], text=True))
    if current != normalized_patch(full_patch.read_text(encoding="utf-8")):
        cpu_patch = EXTRACTOR_DIR / "cpu_device.patch"
        subprocess.run(["git", "-C", str(SAM_REPO), "apply", "--check", str(cpu_patch)], check=True)
        subprocess.run(["git", "-C", str(SAM_REPO), "apply", str(cpu_patch)], check=True)
os.environ["SAM3D_DINOV3_REPO"] = str(DINOV3_REPO)
print("补丁已就绪:", DEVICE)

## 4. 命令行下载 SAM 模型（ModelScope / Hugging Face）

默认执行 ModelScope 1.29.2 的 `python -m modelscope.cli.cli download`；可切换为 `hf download`（通过当前 Python 调用 hf CLI 入口，避免 PATH 指向其他环境）。只下载 `model.ckpt`、`model_config.yaml`、`assets/mhr_model.pt`。

请先阅读 [模型及源许可](https://huggingface.co/facebook/sam-3d-body-dinov3)。使用官方 Hugging Face 时需已获授权，并在终端 `hf auth login` 或 Notebook 平台的 secret 环境变量中配置凭证，不把 token 写入单元格。
下载约 2GB；依赖、缓存和模型建议预留 15–20GB 以上空间。先在临时目录下载、校验，再发布到 CHECKPOINT_DIR；已有目录只校验，不覆盖。

In [ ]:
import json
import shutil
import tempfile
from runtime_checks import MODEL_ID, MODEL_FILES, KNOWN_HASHES, verify_assets

if MODEL_SOURCE not in ("modelscope", "huggingface"):
    raise ValueError("MODEL_SOURCE 必须为 modelscope 或 huggingface")
if CHECKPOINT_DIR.exists():
    sam_assets = verify_assets(CHECKPOINT_DIR, KNOWN_HASHES)
    print("复用已校验模型:", CHECKPOINT_DIR)
else:
    CHECKPOINT_DIR.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(CHECKPOINT_DIR.parent).free < 8 * 1024**3:
        raise RuntimeError("SAM 下载至少需要 8 GiB 可用空间")
    with tempfile.TemporaryDirectory(prefix="sam-cli-", dir=CHECKPOINT_DIR.parent) as temporary:
        staging = Path(temporary) / "checkpoints"
        if MODEL_SOURCE == "modelscope":
            command = [sys.executable, "-m", "modelscope.cli.cli", "download", "--model", MODEL_ID,
                       "--revision", MODEL_REVISION, "--local_dir", str(staging), "--include", *MODEL_FILES]
        else:
            # 等价于 hf download；使用当前 kernel 中安装的 huggingface_hub CLI。
            command = [sys.executable, "-c", "from huggingface_hub.cli.hf import main; main()",
                       "download", MODEL_ID, *MODEL_FILES, "--revision", MODEL_REVISION, "--local-dir", str(staging)]
        print(subprocess.list2cmdline(command))
        download_env = dict(os.environ)
        if MODEL_SOURCE == "modelscope":
            download_env["MODELSCOPE_DOWNLOAD_PARALLELS"] = "8"
        subprocess.run(command, env=download_env, check=True)
        sam_assets = verify_assets(staging, KNOWN_HASHES)
        (staging / "download_provenance.json").write_text(json.dumps(dict(
            source=MODEL_SOURCE, model_id=MODEL_ID, requested_revision=MODEL_REVISION,
            files=sam_assets), indent=2), encoding="utf-8")
        if CHECKPOINT_DIR.exists():
            raise FileExistsError("模型目标目录已出现，停止发布")
        staging.rename(CHECKPOINT_DIR)
print("SAM 模型已就绪:", CHECKPOINT_DIR)

## 5. 下载多人检测器权重（单人模式自动跳过）

从当前 torchvision 的官方权重枚举读取 URL，使用 Python 命令调用 `torch.hub.download_url_to_file` 并检查 URL 中的 SHA256 前缀。已有文件只校验，不覆盖。
[torchvision 代码许可](https://github.com/pytorch/vision/blob/main/LICENSE)与[预训练权重条款](https://github.com/pytorch/vision#pre-trained-model-license)请分别阅读。

In [ ]:
if MULTI_PERSON:
    import re
    import shutil
    import tempfile
    from urllib.parse import urlparse
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights
    from runtime_checks import sha256

    url = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1.url
    parsed = urlparse(url)
    match = re.search(r"-([a-f0-9]{8,64})\.pth$", parsed.path)
    if parsed.scheme != "https" or parsed.hostname != "download.pytorch.org" or not match:
        raise RuntimeError("检测器权重来源或 hash 前缀不符合预期")
    prefix = match.group(1)
    if DETECTOR_WEIGHTS.exists():
        if not DETECTOR_WEIGHTS.is_file() or not sha256(DETECTOR_WEIGHTS).startswith(prefix):
            raise RuntimeError("已有检测器权重校验失败，请检查文件或选择新路径")
    else:
        DETECTOR_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
        if shutil.disk_usage(DETECTOR_WEIGHTS.parent).free < 1024**3:
            raise RuntimeError("检测器下载至少需要 1 GiB 可用空间")
        with tempfile.TemporaryDirectory(prefix="detector-cli-", dir=DETECTOR_WEIGHTS.parent) as temporary:
            candidate = Path(temporary) / "weights.pth"
            command = [sys.executable, "-c",
                       "import sys, torch; torch.hub.download_url_to_file(sys.argv[1], sys.argv[2], hash_prefix=sys.argv[3])",
                       url, str(candidate), prefix]
            print(subprocess.list2cmdline(command))
            subprocess.run(command, check=True)
            if not sha256(candidate).startswith(prefix):
                raise RuntimeError("检测器权重 SHA256 校验失败")
            with candidate.open("rb") as source, DETECTOR_WEIGHTS.open("xb") as target:
                shutil.copyfileobj(source, target)
    print("检测器权重已就绪:", DETECTOR_WEIGHTS)
else:
    print("单人模式，不需要检测器权重")

## 6. 本地校验与模型加载

检查视频、输出路径、源码 commit/补丁、模型文件及 CUDA。GPU 同时加载检测器与 SAM，请先用少量采样帧验证显存和输入。

In [ ]:
from runtime_checks import check_runtime
from extract_video import load_estimator, preflight, preflight_archive
import torch
if not VIDEO.is_file():
    raise FileNotFoundError(VIDEO)
validated_assets = check_runtime(SAM_REPO, DINOV3_REPO, CHECKPOINT_DIR, target=DEVICE, expected_hashes=KNOWN_HASHES)
if MULTI_PERSON:
    if OUTPUT_DIR.exists():
        raise FileExistsError("请使用新的 OUTPUT_DIR，避免混合多次运行")
    if not DETECTOR_WEIGHTS.is_file():
        raise FileNotFoundError("先运行检测器下载单元")
else:
    preflight(VIDEO, OUTPUT_JSON)
    if SAVE_NPZ:
        preflight_archive(OUTPUT_JSON)
estimator, runtime_metadata = load_estimator(SAM_REPO, CHECKPOINT, MHR_MODEL, DEVICE)
runtime_metadata["validated_assets"] = validated_assets
if MULTI_PERSON:
    from multi_person import TorchvisionPersonDetector
    detector = TorchvisionPersonDetector(DETECTOR_WEIGHTS, device=DEVICE,
                                        score_threshold=DETECTOR_THRESHOLD, max_people=MAX_PEOPLE)
print(runtime_metadata)

## 7. 提取视频：多人 / 单人

保留原视频 PTS 时间，不因抽帧或人物晚入场而压缩/归零。多人使用预测 bbox + 贪心 IoU 关联，不是身份重识别；交叉和遮挡可能切换 ID。MAX_MISSING_SECONDS 应大于正常采样间隔。
原始大数组逐帧写 NPZ，JSON 关键点仍占用随帧数增长的内存；长视频请分段并选择新输出路径。

In [ ]:
with torch.inference_mode():
    if MULTI_PERSON:
        from multi_person import extract_multi
        result = extract_multi(VIDEO, OUTPUT_DIR, estimator, detector,
            stride=STRIDE, max_frames=MAX_FRAMES,
            start_seconds=START_SECONDS, end_seconds=END_SECONDS,
            iou_threshold=TRACK_IOU_THRESHOLD, max_missing_seconds=MAX_MISSING_SECONDS,
            inference_type=INFERENCE_TYPE, save_npz=SAVE_NPZ, npz_compressed=NPZ_COMPRESSED,
            continue_on_error=CONTINUE_ON_POSE_ERROR, metadata=runtime_metadata)
        print("状态:", result["status"], "采样帧:", result["sampled_frames"])
        for person in result["tracks"]:
            print("track", person["track_id"], person["status"], person["json_file"])
    else:
        from extract_video import extract
        result = extract(VIDEO, OUTPUT_JSON, estimator, stride=STRIDE, max_frames=MAX_FRAMES,
            start_seconds=START_SECONDS, end_seconds=END_SECONDS, bbox=SINGLE_BBOX,
            inference_type=INFERENCE_TYPE, metadata=runtime_metadata,
            save_npz=SAVE_NPZ, npz_compressed=NPZ_COMPRESSED,
            continue_on_error=CONTINUE_ON_POSE_ERROR)
        print("单人JSON:", OUTPUT_JSON.resolve())

## 8. 检查输出 / 交给 Blender

`motion_multi/track_0001/motion.json` 等文件可逐人交给 `pose-blender`。也可将总 `manifest.json` 交给 `pose-blender-shared`，导入同一场景；详见工程根 README。
共享位置来自未标定的相机相对估计；overview 是展示总览，不是原镜头复原。导出成功请检查 `manifest.complete`；中断归档不能当作完整结果。
只用 `allow_pickle=False` 读取可信 NPZ。

In [ ]:
import json
import numpy as np
if MULTI_PERSON:
    manifest_path = OUTPUT_DIR / "manifest.json"
    completed = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert completed["complete"]
    print("有效人物JSON:", [p["json_file"] for p in completed["tracks"] if p["json_file"]])
    if SAVE_NPZ:
        candidates = OUTPUT_DIR.glob("track_*/raw_npz/frame_*.npz")
        first = next(candidates, None)
        if first is not None:
            with np.load(first, allow_pickle=False) as raw:
                print(first, {k: raw[k].shape for k in raw.files})